# 말로 장부 작성하기 — 코랩 서버

간편장부 사이트(`index.html`)의 🎤 버튼이 보내는 소리를 받아
**Whisper 로 받아적고 → 젬마(로컬 LLM)로 장부 칸을 나눠서** 돌려준다.

- 실행 환경: **코랩 GPU** (런타임 → 런타임 유형 변경 → **T4 GPU**)
- GPU 가 없으면 CPU 로 돌아간다. 코드는 하나다 (느릴 뿐이다)
- 젬마는 **Ollama 로 이 노트북 안에서 돈다. 바깥 API 를 쓰지 않는다**

**위에서부터 차례대로 모두 실행한다.** 맨 아래 칸이 주소 하나를 찍어준다.
그 주소를 사이트의 `AI 주소` 칸에 붙여넣으면 끝이다.


## 1. 설치

In [ ]:
!pip -q install openai-whisper fastapi "uvicorn[standard]" python-multipart nest_asyncio
!apt-get -qq install -y ffmpeg > /dev/null 2>&1
print("설치 끝")

## 2. Whisper 올리기 — GPU 없으면 CPU 로 간다

In [ ]:
import torch, whisper

장치 = "cuda" if torch.cuda.is_available() else "cpu"
print("장치:", 장치)
if 장치 == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("GPU 가 안 잡혔습니다. CPU 로 돕니다 (느립니다).")

소리모델 = whisper.load_model("small", device=장치)
print("Whisper small 올라감")

## 3. 젬마 올리기 — Ollama 를 이 노트북 안에 깐다

바깥 API 를 쓰지 않는다. 모델을 내려받는 데 몇 분 걸린다.

In [ ]:
import os, shutil, subprocess, time, requests

# ── 1) 깔기 ──────────────────────────────────────────────
# 설치 글을 감추지 않는다. 실패하면 왜 실패했는지 보여야 한다.
!curl -fsSL https://ollama.com/install.sh | sh

os.environ["PATH"] = os.environ["PATH"] + ":/usr/local/bin:/usr/bin"
올라마 = shutil.which("ollama")
if not 올라마:
    raise SystemExit("Ollama 가 안 깔렸습니다. 위에 찍힌 설치 글을 보여주세요.")
print("올라마:", 올라마)

# ── 2) 켜기 ──────────────────────────────────────────────
# 이미 떠 있으면 또 띄우지 않는다 (칸을 두 번 실행해도 되게)
def 떴나():
    try:
        requests.get("http://127.0.0.1:11434/api/tags", timeout=2)
        return True
    except Exception:
        return False

if not 떴나():
    기록 = open("/content/ollama.log", "w")
    subprocess.Popen([올라마, "serve"], stdout=기록, stderr=기록,
                     env={**os.environ, "OLLAMA_HOST": "127.0.0.1:11434"})
    for _ in range(90):
        if 떴나():
            break
        time.sleep(1)

if not 떴나():
    print(open("/content/ollama.log", errors="ignore").read()[-3000:])
    raise SystemExit("Ollama 서버가 안 떴습니다. 바로 위 글을 보여주세요.")
print("Ollama 떴다")

# ── 3) 젬마 내려받기 ─────────────────────────────────────
# 4GB 쯤 된다. 몇 분 걸린다. 끊기면 한 번 더 해본다.
for 번째 in range(1, 4):
    print(f"gemma3:4b 내려받는 중… ({번째}번째)")
    결과 = subprocess.run([올라마, "pull", "gemma3:4b"],
                        capture_output=True, text=True)
    if 결과.returncode == 0:
        break
    print(결과.stdout[-1500:])
    print(결과.stderr[-1500:])
    time.sleep(5)
else:
    raise SystemExit("젬마를 못 받았습니다. 바로 위 글을 보여주세요.")

받은것 = requests.get("http://127.0.0.1:11434/api/tags", timeout=10).json()
print("가진 모델:", [하나["name"] for 하나 in 받은것.get("models", [])])

## 4. 말을 장부 칸으로 나누는 규칙

국세청 간편장부 8개 항목에 맞춘다. 규칙은 프롬프트에 적어둔다.

| 정한 것 | 내용 |
|---|---|
| 수입/비용 가르기 | **판 것·받은 것은 수입, 산 것·낸 것은 비용.** 장비·차량처럼 오래 쓰는 것을 사면 자산 |
| 부가세 | **말한 것만 넣는다.** 말하지 않으면 비운다 (멋대로 10% 를 떼지 않는다) |
| 날짜 | "어제", "3월 5일" 같은 말은 오늘 날짜를 기준으로 계산해서 `2026-03-05` 모양으로 |
| 꼭 있어야 하는 칸 | **일자 · 거래내용 · 금액.** 하나라도 없으면 장부에 넣지 않고 빠졌다고 알려준다 |
| 비고 | 음성으로 넣은 줄은 `말` 이라고 적는다 |

In [ ]:
import json, datetime, re

# 장부 한 줄의 칸 이름. 사이트(script.js)가 쓰는 이름과 똑같아야 한다.
칸이름 = ["일자", "계정과목", "거래내용", "거래처",
        "수입금액", "수입부가세", "비용금액", "비용부가세",
        "자산금액", "자산부가세"]

# 이것들이 비면 장부에 넣지 않는다
필수 = {"일자": "날짜", "거래내용": "무엇을 했는지", "금액": "얼마인지"}

지시문 = """너는 한국 개인사업자의 간편장부를 대신 적어주는 사람이다.
사용자가 말한 거래 한 건을 아래 칸으로 나눠서 JSON 하나만 내놓아라. 설명은 쓰지 마라.

칸:
  일자        "2026-03-05" 모양. 오늘은 {오늘}({요일}요일)이다. "어제", "지난주 월요일" 같은 말은 계산해서 넣어라
  계정과목    거래에 맞는 계정과목 하나 (예: 매출, 소모품비, 복리후생비, 차량유지비, 지급수수료, 임차료, 접대비)
  거래내용    무엇을 사고팔았는지 짧게
  거래처      상대 상호나 이름. 말하지 않았으면 ""
  수입금액    판 것·받은 것이면 여기에 숫자. 아니면 ""
  수입부가세  수입일 때 부가세를 말했으면 숫자. 말하지 않았으면 ""
  비용금액    산 것·낸 것이면 여기에 숫자. 아니면 ""
  비용부가세  비용일 때 부가세를 말했으면 숫자. 말하지 않았으면 ""
  자산금액    차량·기계·비품처럼 오래 쓰는 것을 사고팔았으면 여기에 숫자. 아니면 ""
  자산부가세  자산일 때 부가세를 말했으면 숫자. 말하지 않았으면 ""

지켜라:
- 수입금액·비용금액·자산금액 중 **하나에만** 숫자를 넣는다. 나머지 둘은 ""
- 부가세는 **사용자가 말했을 때만** 넣는다. 멋대로 10%를 계산하지 마라
- 금액은 쉼표 없는 숫자로. "삼만원"은 30000, "1만 5천원"은 15000
- 말에 없는 것은 지어내지 말고 "" 로 둬라
- JSON 한 개만 내놓아라

사용자가 한 말: "{말}"
"""

요일이름 = ["월", "화", "수", "목", "금", "토", "일"]

def 숫자로(값):
    """'12,000원' 같은 것도 숫자로 본다. 숫자가 아니면 빈칸."""
    if 값 is None: return ""
    if isinstance(값, bool): return ""
    if isinstance(값, (int, float)): return 값
    글 = re.sub(r"[^0-9.-]", "", str(값))
    try:
        수 = float(글)
    except ValueError:
        return ""
    return int(수) if 수 == int(수) else 수

def 장부줄로(말, 오늘=None):
    """말 한 마디를 장부 한 줄로 나눈다. 돌려주는 것: (줄, 빠진칸)"""
    if not 오늘:
        오늘 = datetime.date.today().isoformat()
    try:
        날 = datetime.date.fromisoformat(오늘)
    except ValueError:
        오늘 = datetime.date.today().isoformat()
        날 = datetime.date.fromisoformat(오늘)

    답 = requests.post("http://127.0.0.1:11434/api/generate", timeout=180, json={
        "model": "gemma3:4b",
        "prompt": 지시문.format(오늘=오늘, 요일=요일이름[날.weekday()], 말=말),
        "format": "json",
        "stream": False,
        "options": {"temperature": 0},
    }).json()

    try:
        생것 = json.loads(답["response"])
    except Exception:
        생것 = {}

    줄 = {키: "" for 키 in 칸이름}
    for 키 in 칸이름:
        값 = 생것.get(키, "")
        줄[키] = 숫자로(값) if 키.endswith(("금액", "부가세")) else str(값 or "").strip()

    # 금액 셋 중 하나만 남긴다. 여러 개가 왔으면 큰 것 하나만 둔다.
    금액칸 = ["수입금액", "비용금액", "자산금액"]
    채워진 = [키 for 키 in 금액칸 if 줄[키] != ""]
    if len(채워진) > 1:
        남길것 = max(채워진, key=lambda 키: 줄[키])
        for 키 in 채워진:
            if 키 != 남길것:
                줄[키] = ""
                줄[키.replace("금액", "부가세")] = ""

    # 날짜 모양이 아니면 버린다 (빠진 것으로 본다)
    if not re.fullmatch(r"\d{4}-\d{2}-\d{2}", str(줄["일자"])):
        줄["일자"] = ""

    빠진칸 = []
    if 줄["일자"] == "": 빠진칸.append(필수["일자"])
    if 줄["거래내용"] == "": 빠진칸.append(필수["거래내용"])
    if all(줄[키] == "" for 키 in 금액칸): 빠진칸.append(필수["금액"])

    줄["비고"] = "말"
    return 줄, 빠진칸

print("규칙 올라감")

## 5. 시험 — 말 한 마디를 글자로 넣어서 확인해본다

In [ ]:
줄, 빠진칸 = 장부줄로("어제 다이소에서 사무용품 3만원어치 샀어")
print(json.dumps(줄, ensure_ascii=False, indent=2))
print("빠진 칸:", 빠진칸 or "없음")

## 6. 서버 켜기

사이트가 소리를 보내면 → Whisper 가 받아적고 → 젬마가 나눠서 → 장부 한 줄로 돌려준다.

In [ ]:
import tempfile, os, threading, time, nest_asyncio, uvicorn
from fastapi import FastAPI, UploadFile, File, Form
from fastapi.middleware.cors import CORSMiddleware

앱 = FastAPI()

# 사이트가 어디에 올라가 있든(깃허브 페이지·로컬 파일) 부를 수 있게 열어둔다.
# 이 서버는 노트북을 닫으면 사라지는 시연용이다.
앱.add_middleware(CORSMiddleware, allow_origins=["*"],
                  allow_methods=["*"], allow_headers=["*"])

@앱.get("/health")
def 살았니():
    return {"ok": True, "장치": 장치}

@앱.post("/voice")
async def 목소리(audio: UploadFile = File(...), today: str = Form(None)):
    끝 = os.path.splitext(audio.filename or "")[1] or ".webm"
    with tempfile.NamedTemporaryFile(suffix=끝, delete=False) as 임시:
        임시.write(await audio.read())
        소리파일 = 임시.name

    시작 = time.time()
    try:
        말 = 소리모델.transcribe(소리파일, language="ko", fp16=(장치 == "cuda"))["text"].strip()
    finally:
        os.unlink(소리파일)
    받아적기초 = round(time.time() - 시작, 2)

    if 말 == "":
        return {"텍스트": "", "줄": None, "빠진칸": ["들린 말"],
                "초": {"받아적기": 받아적기초, "나누기": 0}}

    시작 = time.time()
    줄, 빠진칸 = 장부줄로(말, today)
    나누기초 = round(time.time() - 시작, 2)

    return {"텍스트": 말, "줄": 줄, "빠진칸": 빠진칸,
            "초": {"받아적기": 받아적기초, "나누기": 나누기초}}

nest_asyncio.apply()
threading.Thread(
    target=lambda: uvicorn.run(앱, host="0.0.0.0", port=8000, log_level="warning"),
    daemon=True,
).start()
time.sleep(3)
print("서버 떴다 — 8000번")

## 7. 바깥에서 들어올 주소 만들기

코랩은 바깥에서 바로 못 부른다. 터널을 뚫어 `https://...trycloudflare.com` 주소 하나를 받는다.
**가입도 키도 필요 없다.** 노트북을 닫으면 주소는 사라진다 — 다시 켜면 주소가 바뀐다.

In [ ]:
import subprocess, re, time

!wget -q -O /usr/local/bin/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x /usr/local/bin/cloudflared

기록 = open("/content/tunnel.log", "w")
subprocess.Popen(["cloudflared", "tunnel", "--url", "http://127.0.0.1:8000",
                  "--no-autoupdate"], stdout=기록, stderr=기록)

주소 = ""
for _ in range(60):
    time.sleep(1)
    글 = open("/content/tunnel.log", errors="ignore").read()
    찾음 = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", 글)
    if 찾음:
        주소 = 찾음.group(0)
        break

if 주소:
    print()
    print("=" * 60)
    print("  사이트의 [AI 주소] 칸에 이것을 붙여넣으세요")
    print()
    print("     " + 주소)
    print()
    print("=" * 60)
else:
    print("주소를 못 받았습니다. 이 칸을 다시 실행해보세요.")
    print(open("/content/tunnel.log", errors="ignore").read()[-2000:])

---

## 서버가 사는 동안 이 노트북을 닫지 마세요

시연이 끝나면 런타임을 끊으면 된다. 다음에 쓸 때는 **1번부터 다시** 실행하고,
새로 나온 주소를 사이트에 다시 넣는다.